In [0]:
import pandas as pd

# Ensure DATE is datetime in all three
aero["DATE"] = pd.to_datetime(aero["DATE"])
freight["DATE"] = pd.to_datetime(freight["DATE"])
gscpi["DATE"] = pd.to_datetime(gscpi["DATE"], utc=True).dt.tz_convert(None)

# Create a monthly key (first day of month)
aero["Month"] = aero["DATE"].dt.to_period("M").dt.to_timestamp()
freight["Month"] = freight["DATE"].dt.to_period("M").dt.to_timestamp()
gscpi["Month"] = gscpi["DATE"].dt.to_period("M").dt.to_timestamp()

print(aero[["DATE","Month"]].head())
print(freight[["DATE","Month"]].head())
print(gscpi[["DATE","Month"]].head())

        DATE      Month
0 1972-01-01 1972-01-01
1 1972-02-01 1972-02-01
2 1972-03-01 1972-03-01
3 1972-04-01 1972-04-01
4 1972-05-01 1972-05-01
        DATE      Month
0 2000-02-01 2000-02-01
1 2000-03-01 2000-03-01
2 2000-04-01 2000-04-01
3 2000-05-01 2000-05-01
4 2000-06-01 2000-06-01
        DATE      Month
0 1998-01-31 1998-01-01
1 1998-02-28 1998-02-01
2 1998-03-31 1998-03-01
3 1998-04-30 1998-04-01
4 1998-05-31 1998-05-01


In [0]:
df = (
    aero.drop(columns=["DATE"])
        .merge(freight.drop(columns=["DATE"]), on="Month", how="inner")
        .merge(gscpi.drop(columns=["DATE"]), on="Month", how="inner")
)

display(df.head())
print("Final shape:", df.shape)

Aerospace_Production,Month,Freight_Index,Supply_Pressure
89.0737,2000-02-01T00:00:00.000Z,-1.8,-0.41
88.4736,2000-03-01T00:00:00.000Z,-3.2,-0.28
88.9575,2000-04-01T00:00:00.000Z,-1.9,0.13
88.5102,2000-05-01T00:00:00.000Z,1.0,0.28
86.7047,2000-06-01T00:00:00.000Z,1.0,0.05


Final shape: (310, 4)


In [0]:
df = df.sort_values("Month")

df["Production_Volatility_3mo"] = df["Aerospace_Production"].rolling(3).std()
df["Freight_Volatility_3mo"] = df["Freight_Index"].rolling(3).std()
df["Pressure_Change"] = df["Supply_Pressure"].diff()

display(df.tail(10))

Aerospace_Production,Month,Freight_Index,Supply_Pressure,Production_Volatility_3mo,Freight_Volatility_3mo,Pressure_Change
106.4851,2025-02-01T00:00:00.000Z,0.5,0.0,4.79169722993105,0.45825756949556845,0.18
106.2058,2025-03-01T00:00:00.000Z,0.1,-0.18,5.335938014069837,0.4509249752822736,-0.18
105.9218,2025-04-01T00:00:00.000Z,0.0,-0.24,0.28165326791798445,0.2645751311064321,-0.06
110.478,2025-05-01T00:00:00.000Z,-0.4,0.28,2.5524924838274288,0.26457513110643194,0.52
108.7926,2025-06-01T00:00:00.000Z,0.0,0.09,2.3036573906136355,0.23094010767581924,-0.19000000000000003
107.4701,2025-07-01T00:00:00.000Z,1.3,0.03,1.5075942104322442,0.8888194417315508,-0.06
110.5213,2025-08-01T00:00:00.000Z,0.2,-0.08,1.530099745113206,0.6999999999999899,-0.11
108.482,2025-09-01T00:00:00.000Z,-0.6,-0.02,1.5541614856020691,0.9539392014169381,0.06
101.7806,2025-10-01T00:00:00.000Z,-1.0,-0.08,4.572879161242813,0.611010092660767,-0.06
101.1491,2025-11-01T00:00:00.000Z,1.3,-0.16,4.063639104299915,1.228820572744445,-0.08


In [0]:
out_path = "/Volumes/workspace/default/aerospace_data/final_aerospace_dataset.csv"
df.to_csv(out_path, index=False)
print("Saved:", out_path)

Saved: /Volumes/workspace/default/aerospace_data/final_aerospace_dataset.csv
